# Model v4: Türkiye geneli veri, 8 nesne sınıfı + yol yüzeyi

| | v2 (sunucuda) / v3 telefon | **v4 (bu notebook)** |
|---|---|---|
| Sınıflar | D00, D10, D20, D40 | aynı 4 hasar + **ROGAR** (rögar/ızgara), **PARKE** (bozuk parke taşı), **YAMA** (asfalt yaması), **KALDIRIM** (bozuk kaldırım/bordür) |
| Yüzey | yok | ayrı sınıflandırıcı: **asfalt / parke / beton / toprak** |
| Türkiye verisi | 598 sağlam + 87 elle etiketli | + **81 ilden ~10.000 Mapillary görüntüsü** (Bursa hariç: test için ayrıldı) |
| Etiketleme | elle | **otomatik** (v2 + Grounding DINO + CLIP) ve **Claude'un görsel incelemesi** |

**Neden ROGAR ve YAMA?** Bursa'daki yanlış alarmların başında "rögar → çukur" geliyordu. Model ayrı bir sınıf öğrenince ikisini ayırabilir; yama da hasarla karışmaz.

## Akış
**Bölüm A (1–6):** Görüntüleri toplar ve otomatik etiketler. Sonra **inceleme sayfalarını** Drive'a (`v4/inceleme/`) yazar ve durur.
→ Claude'a "v4 inceleme hazır" yazın. Claude sayfalara bakıp kabul/red kararlarını yazar, siz de bu kararları **7. hücreye** yapıştırırsınız.

**Bölüm B (7–11):** Kararlara göre veri setini kurar. 7. hücrede seçilen sınıflarla ana model (YOLO11m, 800), telefon modeli (YOLO11n, 640) ve yüzey sınıflandırıcısını eğitir, ardından ölçer.

## Çalıştırma
1. Mapillary anahtarı: soldaki 🔑 (Gizli anahtarlar) → `MAPILLARY_TOKEN` adıyla ekleyin ve "Not defteri erişimi"ni açın.
2. Drive'daki `yol-hasar-tespit` klasöründe şunlar olsun: `v2/best_v2.pt`, `v2/esikler_v2.json`, `v3/best_canli.pt`, `turkiye_egitim.zip`, `hasar_etiket.zip`.
3. `Çalışma zamanı → Çalışma zamanı türünü değiştir → L4 GPU`, sonra `Tümünü çalıştır`.

Oturum koparsa yeniden `Tümünü çalıştır` yeterli. Her adım sonucunu Drive'a yazar ve kaldığı yerden devam eder.
Bölüm A yaklaşık 2 saat, Bölüm B'deki eğitimler ise 6–8 saat sürer (gece çalıştırın).

> **İnceleme sonucu (2026-11-11):** Grounding DINO yeni sınıfları Türkiye sokak görüntülerinde bulamadı. Doğru aday sayıları: rögar 150'de ~30, bozuk parke 450'de 1, asfalt yaması 300'de 1, bozuk kaldırım 300'de 0. Bu yüzden `YENI:` satırı boş bırakıldı: v4, dört hasar sınıfı (D00–D40) ve yeni Türkiye görüntüleriyle eğitilir. Hasar etiketlerinden yalnızca D00 ve D40 güvenilir bulundu; v2'nin D10/D20 kutusu olan görüntüler atlanır. Yol yüzeyi sınıflandırıcısı asfalt ve parke ile eğitilir.

## 1. Ortam

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU YOK! Çalışma zamanı → Çalışma zamanı türünü değiştir → L4 GPU, sonra baştan çalıştırın.'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive, userdata
drive.mount('/content/drive')
import os
from pathlib import Path
DRIVE_DIR = '/content/drive/MyDrive/yol-hasar-tespit'
OUT = f'{DRIVE_DIR}/v4'
WORK = Path('/content/v4')
DEVICE = 'cuda'
MLY_TOKEN = userdata.get('MAPILLARY_TOKEN')

!git clone -q https://github.com/mahmutsamiseyhan/yol-hasar-tespit.git /content/yol-hasar-tespit || (cd /content/yol-hasar-tespit && git pull -q)
%cd /content/yol-hasar-tespit
!pip install -q -U ultralytics remotezip onnx onnxslim "transformers>=4.51" accelerate

In [ ]:
import os, json, csv, random, zipfile, shutil, time, io
from collections import Counter
from pathlib import Path
import numpy as np, cv2
from PIL import Image

SINIFLAR = ['D00', 'D10', 'D20', 'D40', 'ROGAR', 'PARKE', 'YAMA', 'KALDIRIM']
SINIF_AD = {'D00': 'Boyuna çatlak', 'D10': 'Enine çatlak', 'D20': 'Timsah sırtı', 'D40': 'Çukur',
            'ROGAR': 'Rögar kapağı / ızgara', 'PARKE': 'Bozuk parke taşı', 'YAMA': 'Asfalt yaması',
            'KALDIRIM': 'Bozuk kaldırım / bordür'}
HASAR = SINIFLAR[:4]
YENI = SINIFLAR[4:]
YUZEYLER = ['asfalt', 'parke', 'beton', 'toprak']

HAM = WORK / 'tr_genel' / 'images'        # toplanan görüntüler (hızlı yerel disk)
ETIKET = Path(OUT) / 'etiket'             # otomatik etiket ara sonuçları (Drive: oturum koparsa kaybolmaz)
INCELEME = Path(OUT) / 'inceleme'         # Claude'un bakacağı sayfalar
for p in (HAM, ETIKET, INCELEME, Path(OUT) / 'ham'):
    p.mkdir(parents=True, exist_ok=True)

def jload(p, default=None):
    p = Path(p)
    return json.load(open(p, encoding='utf-8')) if p.exists() else default

def jsave(obj, p):
    tmp = Path(str(p) + '.tmp')
    json.dump(obj, open(tmp, 'w', encoding='utf-8'), ensure_ascii=False)
    os.replace(tmp, p)

## 2. Türkiye geneli görüntü toplama (81 il, Bursa hariç)

- Her ilin merkezi çevresinde 4 büyük kutu (~20×20 km) sorgulanır. Sunucu zaman aşımına düşerse kutu otomatik olarak küçük parçalara bölünür.
- Görüntüler arasında en az ~100 m bırakılır ve her ~1 km'lik kareden en fazla 4 görüntü alınır. Böylece aynı sokak tekrar etmez.
- **Seçimi model yapmaz:** havuz rastgeledir. Karanlık görüntüler ve önceki eğitim paketlerinde zaten olan görüntüler atılır.
- Her il ayrı bir zip olarak Drive'a (`v4/ham/`) yazılır. Hücre yeniden çalıştırılırsa hazır iller atlanır.

In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

PER_PROVINCE = 125
API = 'https://graph.mapillary.com/images'
FIELDS = 'id,thumb_1024_url,computed_geometry,geometry,captured_at,creator,is_pano'
BURSA_IL = (39.55, 28.10, 40.70, 29.95)   # güney, batı, kuzey, doğu: Bursa test için ayrıldı
MIN_PARLAKLIK = 60
PROVINCES = {
    'adana': (37.00, 35.32), 'adiyaman': (37.76, 38.28), 'afyonkarahisar': (38.76, 30.54), 'agri': (39.72, 43.05),
    'amasya': (40.65, 35.83), 'ankara_merkez': (39.93, 32.85), 'ankara_bati': (39.95, 32.70),
    'antalya': (36.89, 30.71), 'artvin': (41.18, 41.82), 'aydin': (37.85, 27.84), 'balikesir': (39.65, 27.89),
    'bilecik': (40.14, 29.98), 'bingol': (38.88, 40.50), 'bitlis': (38.40, 42.11), 'bolu': (40.73, 31.61),
    'burdur': (37.72, 30.29), 'canakkale': (40.15, 26.41), 'cankiri': (40.60, 33.62), 'corum': (40.55, 34.95),
    'denizli': (37.78, 29.09), 'diyarbakir': (37.92, 40.21), 'edirne': (41.68, 26.56), 'elazig': (38.68, 39.22),
    'erzincan': (39.75, 39.49), 'erzurum': (39.90, 41.27), 'eskisehir': (39.78, 30.52), 'gaziantep': (37.07, 37.38),
    'giresun': (40.91, 38.39), 'gumushane': (40.46, 39.48), 'hakkari': (37.58, 43.74), 'hatay': (36.20, 36.16),
    'isparta': (37.76, 30.55), 'mersin': (36.80, 34.63), 'istanbul_avrupa': (41.05, 28.90),
    'istanbul_anadolu': (40.99, 29.10), 'istanbul_bati': (41.02, 28.70), 'izmir_merkez': (38.42, 27.14),
    'izmir_kuzey': (38.48, 27.08), 'kars': (40.60, 43.10), 'kastamonu': (41.38, 33.78), 'kayseri': (38.72, 35.49),
    'kirklareli': (41.73, 27.22), 'kirsehir': (39.15, 34.16), 'kocaeli': (40.77, 29.92), 'konya': (37.87, 32.48),
    'kutahya': (39.42, 29.98), 'malatya': (38.35, 38.31), 'manisa': (38.61, 27.43), 'kahramanmaras': (37.58, 36.94),
    'mardin': (37.31, 40.74), 'mugla': (37.22, 28.36), 'mus': (38.75, 41.51), 'nevsehir': (38.62, 34.71),
    'nigde': (37.97, 34.68), 'ordu': (40.98, 37.88), 'rize': (41.02, 40.52), 'sakarya': (40.78, 30.40),
    'samsun': (41.29, 36.33), 'siirt': (37.93, 41.94), 'sinop': (42.03, 35.15), 'sivas': (39.75, 37.02),
    'tekirdag': (40.98, 27.51), 'tokat': (40.31, 36.55), 'trabzon': (41.00, 39.72), 'tunceli': (39.11, 39.55),
    'sanliurfa': (37.16, 38.79), 'usak': (38.68, 29.41), 'van': (38.49, 43.38), 'yozgat': (39.82, 34.81),
    'zonguldak': (41.45, 31.79), 'aksaray': (38.37, 34.03), 'bayburt': (40.26, 40.23), 'karaman': (37.18, 33.22),
    'kirikkale': (39.85, 33.51), 'batman': (37.88, 41.13), 'sirnak': (37.52, 42.46), 'bartin': (41.63, 32.34),
    'ardahan': (41.11, 42.70), 'igdir': (39.92, 44.05), 'yalova': (40.65, 29.27), 'karabuk': (41.20, 32.62),
    'kilis': (36.72, 37.12), 'osmaniye': (37.07, 36.25), 'duzce': (40.84, 31.16),
}

def bursada(lat, lon):
    s, w, n, e = BURSA_IL
    return s <= lat <= n and w <= lon <= e

def mly_query(bbox, limit=1000):
    w, s, e, n = bbox
    params = dict(access_token=MLY_TOKEN, fields=FIELDS, bbox=f'{w:.5f},{s:.5f},{e:.5f},{n:.5f}', limit=limit,
                  is_pano='false', start_captured_at='2016-01-01T00:00:00Z')
    r = requests.get(API, params=params, timeout=25)
    r.raise_for_status()
    return r.json().get('data', [])

def il_adaylari(lat, lon, gerekli, rng, d=0.09, sure=180):
    # Merkez çevresinde 4 büyük kutu; sunucu zaman aşımına düşerse (yoğun şehir) kutu 4'e bölünür.
    # Yeterli aday bulununca ya da il başına süre dolunca durur.
    kuyruk = [((lon - d, lat - d, lon, lat), 0), ((lon, lat - d, lon + d, lat), 0),
              ((lon - d, lat, lon, lat + d), 0), ((lon, lat, lon + d, lat + d), 0)]
    rng.shuffle(kuyruk)
    adaylar, t0 = [], time.time()
    while kuyruk and len(adaylar) < gerekli and time.time() - t0 < sure:
        (w, s, e, n), derinlik = kuyruk.pop(0)
        try:
            adaylar += mly_query((w, s, e, n))
        except Exception:
            if derinlik < 3:
                mx, my = (w + e) / 2, (s + n) / 2
                parcalar = [((w, s, mx, my), derinlik + 1), ((mx, s, e, my), derinlik + 1),
                            ((w, my, mx, n), derinlik + 1), ((mx, my, e, n), derinlik + 1)]
                rng.shuffle(parcalar)
                kuyruk += parcalar
    return adaylar

def onceki_kimlikler():
    ids = set()
    for z in ['turkiye_egitim.zip', 'hasar_etiket.zip']:
        p = Path(DRIVE_DIR) / z
        if not p.exists():
            continue
        with zipfile.ZipFile(p) as zf:
            for n in zf.namelist():
                stem = Path(n).stem
                if n.startswith('images/') and '_' in stem:
                    ids.add(stem.rsplit('_', 1)[-1])
    return ids

def indir(url):
    for i in range(3):
        try:
            data = requests.get(url, timeout=40).content
            img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_GRAYSCALE)
            return data if img is not None and img.mean() >= MIN_PARLAKLIK else None
        except Exception:
            time.sleep(1 + i)
    return None

def il_topla(il, merkez, rng, atla):
    adaylar = il_adaylari(merkez[0], merkez[1], gerekli=PER_PROVINCE * 15, rng=rng)
    rng.shuffle(adaylar)
    secilen, k100, k1km = [], set(), {}
    for im in adaylar:
        g = (im.get('computed_geometry') or im.get('geometry') or {}).get('coordinates')
        if not g or not im.get('thumb_1024_url') or im['id'] in atla:
            continue
        lo, la = g
        a, b = (round(la, 3), round(lo, 3)), (round(la, 2), round(lo, 2))
        if bursada(la, lo) or a in k100 or k1km.get(b, 0) >= 4:
            continue
        k100.add(a); k1km[b] = k1km.get(b, 0) + 1
        secilen.append((im, la, lo))
        if len(secilen) >= int(PER_PROVINCE * 1.4):   # karanlıklar atılacak: fazladan al
            break
    if not secilen:   # sorgu başarısız ya da bölgede görüntü yok: zip yazılmaz, sonraki çalıştırmada yeniden denenir
        return 0, len(adaylar)
    with ThreadPoolExecutor(16) as ex:
        veriler = list(ex.map(lambda t: indir(t[0]['thumb_1024_url']), secilen))
    satirlar, buf = [], io.BytesIO()
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_STORED) as zf:
        for (im, la, lo), data in zip(secilen, veriler):
            if data is None or len(satirlar) >= PER_PROVINCE:
                continue
            ad = f'{il}_{im["id"]}.jpg'
            zf.writestr(ad, data)
            atla.add(im['id'])
            satirlar.append({'dosya': ad, 'il': il, 'enlem': round(la, 6), 'boylam': round(lo, 6),
                             'cekim_tarihi': datetime.fromtimestamp(im['captured_at'] / 1000, timezone.utc).date().isoformat(),
                             'kullanici': (im.get('creator') or {}).get('username', ''),
                             'mapillary_url': f'https://www.mapillary.com/app/?pKey={im["id"]}', 'lisans': 'CC BY-SA 4.0'})
        s = io.StringIO()
        w = csv.DictWriter(s, fieldnames=list(satirlar[0]) if satirlar else ['dosya'])
        w.writeheader(); w.writerows(satirlar)
        zf.writestr('atif.csv', s.getvalue())
    tmp = Path(OUT) / 'ham' / f'{il}.zip.tmp'
    tmp.write_bytes(buf.getvalue())
    os.replace(tmp, Path(OUT) / 'ham' / f'{il}.zip')
    return len(satirlar), len(adaylar)

rng = random.Random(2026)
atla = onceki_kimlikler()
for il, merkez in PROVINCES.items():
    hedef = Path(OUT) / 'ham' / f'{il}.zip'
    if hedef.exists():
        with zipfile.ZipFile(hedef) as zf:
            atla.update(Path(n).stem.rsplit('_', 1)[-1] for n in zf.namelist() if n.endswith('.jpg'))
        continue
    n, aday = il_topla(il, merkez, rng, atla)
    print(f'{il:18s} {n:4d} görüntü  ({aday} aday)', flush=True)

# Hepsini yerel diske aç + tek atıf dosyası
atif = []
for z in sorted((Path(OUT) / 'ham').glob('*.zip')):
    with zipfile.ZipFile(z) as zf:
        for n in zf.namelist():
            if n.endswith('.jpg') and not (HAM / n).exists():
                zf.extract(n, HAM)
        atif += list(csv.DictReader(io.StringIO(zf.read('atif.csv').decode('utf-8'))))
with open(HAM.parent / 'atif.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(atif[0])); w.writeheader(); w.writerows(atif)
IL_OF = {r['dosya']: r['il'] for r in atif}
GORUNTULER = sorted(p for p in HAM.glob('*.jpg') if p.name in IL_OF)
print(f'\nToplam {len(GORUNTULER)} görüntü, {len(set(IL_OF.values()))} il/merkez')

## 3. Hasar etiketleri (D00–D40): v2 modeli

v2, Bursa'da sağlam yolda yanlış alarm vermeyen modelimiz. Burada **yalnızca emin olduğu** kutular etiket olarak alınır: güven, sınıf eşiğinin en az 0.10 üstünde ve en az 0.40 olmalı.
Eşiğin hemen altında kalan şüpheli kutusu olan görüntüler eğitime girmez. Böylece modele "burada hasar yok" diye yanlış bir şey öğretilmez.
Not: Bu yöntem hasar sınıflarında v2'nin zaten bildiğini pekiştirir. Asıl kazanç yeni sınıflardan ve Türkiye yollarının çeşitliliğinden gelir.

In [ ]:
from ultralytics import YOLO

V2 = YOLO(f'{DRIVE_DIR}/v2/best_v2.pt')
ESIK_V2 = jload(f'{DRIVE_DIR}/v2/esikler_v2.json')['esikler']

def hasar_etiketle(yollar, onbellek, parca=256):
    sonuc = jload(onbellek, {})
    kalan = [p for p in yollar if p.name not in sonuc]
    for i in range(0, len(kalan), parca):
        grup = kalan[i:i + parca]
        # Liste verilince Ultralytics hepsini TEK batch yapar (batch= yok sayılır) -> 16'lık parçalarla çağır
        sonuclar = [r for j in range(0, len(grup), 16)
                    for r in V2.predict([str(x) for x in grup[j:j + 16]], imgsz=800, conf=0.10, half=True, verbose=False)]
        for p, r in zip(grup, sonuclar):
            kesin, belirsiz = [], 0
            for b, c, k in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist(), r.boxes.cls.tolist()):
                ad = r.names[int(k)]
                t = ESIK_V2.get(ad, 0.35)
                if c >= max(t + 0.10, 0.40):
                    kesin.append([ad, round(c, 3), [round(v, 1) for v in b]])
                elif c >= t - 0.05:
                    belirsiz += 1
            sonuc[p.name] = {'kesin': kesin, 'belirsiz': belirsiz, 'boyut': [r.orig_shape[1], r.orig_shape[0]]}
        jsave(sonuc, onbellek)
        print(f'  hasar: {len(sonuc)}/{len(yollar)}', flush=True)
    return sonuc

torch.cuda.empty_cache()
HASAR_ET = hasar_etiketle(GORUNTULER, ETIKET / 'hasar_tr_genel.json')
say = {k: sum(1 for v in HASAR_ET.values() for b in v['kesin'] if b[0] == k) for k in HASAR}
print('Kesin hasar kutuları:', say, '| şüpheli görüntü:', sum(1 for v in HASAR_ET.values() if v['belirsiz']))

## 4. Yeni sınıf adayları: Grounding DINO (metinle nesne bulma)

Model her görüntüde şu ifadeleri arar: "manhole cover", "storm drain grate", "damaged paving stones", "sunken cobblestone pavement", "asphalt patch", "broken curb".
Bulunan kutular **aday** olarak kalır, hemen etiket olmaz: rögar dışındaki sınıflarda bu tür modellerin isabeti düşüktür.
Gökyüzü bölgesindeki (görüntünün üst %35'i) adaylar ile çok küçük veya çok büyük kutular elenir.

In [ ]:
from torchvision.ops import nms
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

GD_ID = 'IDEA-Research/grounding-dino-base'
gd_proc = AutoProcessor.from_pretrained(GD_ID)
gd = AutoModelForZeroShotObjectDetection.from_pretrained(GD_ID).to(DEVICE).eval()
PROMPT = 'manhole cover. storm drain grate. damaged paving stones. sunken cobblestone pavement. asphalt patch. broken curb.'
ANAHTAR = [('manhole', 'ROGAR'), ('drain', 'ROGAR'), ('grate', 'ROGAR'), ('curb', 'KALDIRIM'), ('patch', 'YAMA'),
           ('paving', 'PARKE'), ('cobblestone', 'PARKE'), ('stone', 'PARKE'), ('pavement', 'PARKE')]
GD_BF16 = [DEVICE == 'cuda']

def sinif_bul(metin):
    m = (metin or '').lower()
    return next((s for k, s in ANAHTAR if k in m), None)

def gd_post(out, inputs, boyutlar, esik):
    kw = dict(text_threshold=0.25, target_sizes=boyutlar)
    try:
        return gd_proc.post_process_grounded_object_detection(out, inputs['input_ids'], threshold=esik, **kw)
    except TypeError:   # eski transformers sürümleri
        return gd_proc.post_process_grounded_object_detection(out, inputs['input_ids'], box_threshold=esik, **kw)

def gd_calistir(imgs):
    inputs = gd_proc(images=imgs, text=[PROMPT] * len(imgs), return_tensors='pt', padding=True).to(DEVICE)
    with torch.inference_mode():
        if GD_BF16[0]:
            try:
                with torch.autocast('cuda', dtype=torch.bfloat16):
                    return gd(**inputs), inputs
            except Exception as e:
                print('bf16 desteklenmedi, fp32 ile devam:', e); GD_BF16[0] = False
        return gd(**inputs), inputs

def yeni_aday(yollar, onbellek, esik=0.30, batch=8):
    sonuc = jload(onbellek, {})
    kalan = [p for p in yollar if p.name not in sonuc]
    for i in range(0, len(kalan), batch):
        grup = kalan[i:i + batch]
        imgs = [Image.open(p).convert('RGB') for p in grup]
        out, inputs = gd_calistir(imgs)
        out.logits = out.logits.float(); out.pred_boxes = out.pred_boxes.float()
        for p, im, r in zip(grup, imgs, gd_post(out, inputs, [im.size[::-1] for im in imgs], esik)):
            W, H = im.size
            etiketler = r.get('text_labels', r.get('labels'))
            kutular, skorlar, siniflar = [], [], []
            for b, s, l in zip(r['boxes'].float().cpu().tolist(), r['scores'].float().cpu().tolist(), etiketler):
                k = sinif_bul(l if isinstance(l, str) else '')
                x1, y1, x2, y2 = b
                alan = (x2 - x1) * (y2 - y1) / (W * H)
                if k is None or not (0.0004 <= alan <= 0.25) or (y1 + y2) / 2 < 0.35 * H:
                    continue
                kutular.append(b); skorlar.append(s); siniflar.append(k)
            tut = []
            for k in set(siniflar):
                idx = [j for j, c in enumerate(siniflar) if c == k]
                keep = nms(torch.tensor([kutular[j] for j in idx]), torch.tensor([skorlar[j] for j in idx]), 0.5)
                tut += [idx[j] for j in keep.tolist()]
            sonuc[p.name] = [[siniflar[j], round(skorlar[j], 3), [round(v, 1) for v in kutular[j]]] for j in tut]
        if (i // batch) % 50 == 0 or i + batch >= len(kalan):
            jsave(sonuc, onbellek)
            print(f'  yeni sınıf: {len(sonuc)}/{len(yollar)}', flush=True)
    return sonuc

YENI_ADAY = yeni_aday(GORUNTULER, ETIKET / 'yeni_tr_genel.json')
print('Adaylar:', {k: sum(1 for v in YENI_ADAY.values() for b in v if b[0] == k) for k in YENI})

## 5. Yol yüzeyi: CLIP

Görüntünün aracın hemen önündeki yol bölgesi (alt orta kısım) CLIP ile dört türden birine atanır: asfalt / parke / beton / toprak.
Olasılığı 0.55'in altında kalan görüntüler "belirsiz" sayılır ve yüzey eğitimine girmez.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_ID = 'openai/clip-vit-large-patch14'
clip = CLIPModel.from_pretrained(CLIP_ID).to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained(CLIP_ID)
YUZEY_METIN = {
    'asfalt': ['a photo of an asphalt road', 'a street with asphalt pavement', 'a highway with asphalt surface'],
    'parke': ['a street paved with interlocking paving stones', 'a cobblestone street', 'a road paved with concrete pavers'],
    'beton': ['a concrete road', 'a road made of concrete slabs'],
    'toprak': ['a dirt road', 'an unpaved gravel road', 'a muddy rural track'],
}

def yol_kirp(im):
    W, H = im.size
    return im.crop((int(0.2 * W), int(0.55 * H), int(0.8 * W), int(0.95 * H)))

def _ozellik(f):
    if not torch.is_tensor(f):   # yeni transformers sürümleri çıktı nesnesi döndürebilir
        f = next(v for v in (getattr(f, 'image_embeds', None), getattr(f, 'text_embeds', None), getattr(f, 'pooler_output', None))
                 if v is not None)
    return f / f.norm(dim=-1, keepdim=True)

with torch.inference_mode():
    metinler = [m for y in YUZEYLER for m in YUZEY_METIN[y]]
    t = _ozellik(clip.get_text_features(**clip_proc(text=metinler, return_tensors='pt', padding=True).to(DEVICE)))
    TXT, j = [], 0
    for y in YUZEYLER:
        n = len(YUZEY_METIN[y]); v = t[j:j + n].mean(0); TXT.append(v / v.norm()); j += n
    TXT = torch.stack(TXT)

def yuzey_etiketle(yollar, onbellek, batch=64):
    sonuc = jload(onbellek, {})
    kalan = [p for p in yollar if p.name not in sonuc]
    for i in range(0, len(kalan), batch):
        grup = kalan[i:i + batch]
        imgs = [yol_kirp(Image.open(p).convert('RGB')) for p in grup]
        with torch.inference_mode():
            f = _ozellik(clip.get_image_features(**clip_proc(images=imgs, return_tensors='pt').to(DEVICE)))
            olas = (clip.logit_scale.exp() * f @ TXT.T).softmax(-1).float().cpu().numpy()
        for p, o in zip(grup, olas):
            sonuc[p.name] = [YUZEYLER[int(o.argmax())], round(float(o.max()), 3)]
        if (i // batch) % 20 == 0 or i + batch >= len(kalan):
            jsave(sonuc, onbellek)
    return sonuc

YUZEY_ET = yuzey_etiketle(GORUNTULER, ETIKET / 'yuzey_tr_genel.json')
print('Yüzey (olasılık ≥ 0.55):', Counter(v[0] for v in YUZEY_ET.values() if v[1] >= 0.55),
      '| belirsiz:', sum(1 for v in YUZEY_ET.values() if v[1] < 0.55))

## 6. İnceleme sayfaları (Claude için)

Her sayfada numaralı 30 kırpılmış aday var. Sarı çerçeve, adayın kendisidir.
- **Yeni sınıflar:** Her sınıf için adayların yarısı en yüksek skorlulardan, yarısı geri kalandan **rastgele** seçilir. Rastgele yarı, hangi skorun üstündeki incelenmemiş adayların otomatik kabul edilebileceğini hesaplamak için kullanılır (Bölüm B, 7. hücre).
- **Hasar örnekleri:** Her hasar sınıfından 30 rastgele kesin kutu. Amaç v2'nin Türkiye yollarındaki isabetini ölçmek.
- **Yüzey:** Her türden 48 rastgele görüntü. Amaç CLIP'in doğruluğunu ölçmek.

Sayfalar `v4/inceleme/` klasörüne yazılır. Bu klasör paylaşılan `yol-hasar-tespit` klasörünün içinde olduğu için Claude aynı bağlantıdan görebilir.

In [ ]:
from PIL import ImageDraw, ImageFont

HUCRE, KOL, SATIR = 224, 6, 5
INCELE_SAYI = {'ROGAR': 150, 'PARKE': 450, 'YAMA': 300, 'KALDIRIM': 300}
try:
    FONT = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 22)
    FONT_K = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 18)
except OSError:
    FONT = FONT_K = ImageFont.load_default()

def kirp(dosya, kutu):
    im = Image.open(HAM / dosya).convert('RGB')
    W, H = im.size
    x1, y1, x2, y2 = kutu
    m = 0.6 * max(x2 - x1, y2 - y1, 40)
    cx1, cy1, cx2, cy2 = max(0, x1 - m), max(0, y1 - m), min(W, x2 + m), min(H, y2 + m)
    c = im.crop((int(cx1), int(cy1), int(cx2), int(cy2)))
    s = HUCRE / max(c.size)
    c = c.resize((max(1, int(c.size[0] * s)), max(1, int(c.size[1] * s))))
    ImageDraw.Draw(c).rectangle([(x1 - cx1) * s, (y1 - cy1) * s, (x2 - cx1) * s, (y2 - cy1) * s], outline=(255, 220, 0), width=2)
    return c

def sayfa(hucreler, baslik, dosya, kol=KOL, satir=SATIR):
    tuval = Image.new('RGB', (kol * HUCRE, satir * HUCRE + 34), (25, 25, 25))
    d = ImageDraw.Draw(tuval)
    d.text((8, 6), baslik, fill=(255, 255, 255), font=FONT_K)
    for i, c in enumerate(hucreler):
        x, y = (i % kol) * HUCRE, 34 + (i // kol) * HUCRE
        tuval.paste(c, (x + (HUCRE - c.size[0]) // 2, y + (HUCRE - c.size[1]) // 2))
        d.rectangle([x, y, x + 34, y + 28], fill=(0, 0, 0))
        d.text((x + 4, y + 2), str(i + 1), fill=(255, 255, 255), font=FONT)
        d.rectangle([x, y, x + HUCRE - 1, y + HUCRE - 1], outline=(70, 70, 70))
    tuval.save(INCELEME / dosya, quality=85)

def inceleme_hazirla():
    indeks = {'sayfalar': {}, 'yuzey': {}}
    r = random.Random(7)
    for k in YENI:
        adaylar = sorted([(d, *b[1:]) for d, v in YENI_ADAY.items() for b in v if b[0] == k], key=lambda t: -t[1])
        n = INCELE_SAYI[k]
        ust, kalan = adaylar[:n // 2], adaylar[n // 2:]
        secim = [(a, 'ust') for a in ust] + [(a, 'rastgele') for a in r.sample(kalan, min(len(kalan), n - len(ust)))]
        secim.sort(key=lambda t: -t[0][1])
        for s in range(0, len(secim), KOL * SATIR):
            parca = secim[s:s + KOL * SATIR]
            ad = f'{k}_{s // (KOL * SATIR) + 1:02d}'
            sayfa([kirp(a[0], a[2]) for a, _ in parca], f'{ad}  ({SINIF_AD[k]})  skor {parca[0][0][1]:.2f}-{parca[-1][0][1]:.2f}', ad + '.jpg')
            indeks['sayfalar'][ad] = [{'sinif': k, 'dosya': a[0], 'skor': a[1], 'kutu': a[2], 'grup': g} for a, g in parca]
    for k in HASAR:   # v2'nin Türkiye'deki isabeti
        kutular = [(d, b) for d, v in HASAR_ET.items() for b in v['kesin'] if b[0] == k]
        parca = r.sample(kutular, min(30, len(kutular)))
        if parca:
            ad = f'HASAR_{k}'
            sayfa([kirp(d, b[2]) for d, b in parca], f'{ad}  ({SINIF_AD[k]}, v2 kesin kutuları, rastgele)', ad + '.jpg')
            indeks['sayfalar'][ad] = [{'sinif': k, 'dosya': d, 'skor': b[1], 'kutu': b[2], 'grup': 'ornek'} for d, b in parca]
    for y in YUZEYLER:
        adlar = [d for d, v in YUZEY_ET.items() if v[0] == y and v[1] >= 0.55]
        parca = r.sample(adlar, min(48, len(adlar)))
        if parca:
            hucreler = []
            for d in parca:
                c = yol_kirp(Image.open(HAM / d).convert('RGB'))
                s = HUCRE / max(c.size); hucreler.append(c.resize((int(c.size[0] * s), int(c.size[1] * s))))
            ad = f'YUZEY_{y}'
            sayfa(hucreler, f'{ad}  (CLIP: {y}, rastgele 48)', ad + '.jpg', kol=8, satir=6)
            indeks['yuzey'][ad] = parca
    jsave(indeks, INCELEME / 'indeks.json')
    return indeks

if not (INCELEME / 'indeks.json').exists():
    inceleme_hazirla()
INDEKS = jload(INCELEME / 'indeks.json')
print(len(INDEKS['sayfalar']), 'aday sayfası +', len(INDEKS['yuzey']), 'yüzey sayfası ->', INCELEME)
print('\n>>> Claude\'a "v4 inceleme hazır" yazın. Kararları aşağıdaki 7. hücreye yapıştırıp Tümünü çalıştır deyin.')

---
# Bölüm B: İnceleme kararları → veri seti → eğitim

## 7. İnceleme kararları (Claude yazar)

Claude kararları aşağıdaki `KARARLAR` metnine yapıştırılacak biçimde verir. İsterseniz aynı metni Drive'a `v4/inceleme/kararlar.txt` olarak da koyabilirsiniz; dosya varsa öncelik onundur.

| Satır | Anlamı |
|---|---|
| `ROGAR_01: hepsi` | sayfadaki bütün adaylar doğru |
| `PARKE_03: red 2,7,15` | listelenenler yanlış, gerisi doğru |
| `YAMA_02: kabul 4,9` | yalnızca listelenenler doğru |
| `KALDIRIM_05: hicbiri` | hiçbiri doğru değil |
| `HASAR_D00: red 3,8` | v2'nin hasar kutuları için aynı biçim |
| `HASAR: D00, D10, D20, D40` | Türkiye'de etiketi güvenilir bulunan hasar sınıfları |
| `YUZEY: asfalt, parke, toprak` | CLIP etiketi güvenilir bulunan yüzey türleri |

**İncelenmeyen adaylar:** Rastgele örneklenen adaylarda isabetin en az %85 olduğu en düşük skor hesaplanır. Bu skorun üstündeki adaylar otomatik kabul edilir. Altında kalan ama skoru 0.40'ı geçen bir aday varsa, o görüntü eğitime girmez.

In [ ]:
KARARLAR = '''
# Claude incelemesi, 2026-11-11. Rakamlar sayfadaki kutu numaraları.
# Rögar: üst skorlarda ~%40, rastgelede ~%7 doğru. Çoğu yanlış aday araç torpidosu/kaputu.
ROGAR_01: kabul 3,5,7,8,13,15,16,18,20,21,22,26,30
ROGAR_02: kabul 2,5,7,9,19,25,27,28
ROGAR_03: kabul 10,22
ROGAR_04: kabul 2,4,5,13,21
ROGAR_05: hicbiri
# Parke: adaylar taş duvar, toprak, yol kenarı. 450 adayda 1 gerçek bozuk parke.
PARKE_01: hicbiri
PARKE_02: hicbiri
PARKE_03: hicbiri
PARKE_04: hicbiri
PARKE_05: hicbiri
PARKE_06: hicbiri
PARKE_07: hicbiri
PARKE_08: hicbiri
PARKE_09: hicbiri
PARKE_10: hicbiri
PARKE_11: hicbiri
PARKE_12: hicbiri
PARKE_13: hicbiri
PARKE_14: hicbiri
PARKE_15: kabul 11
# Yama: model yolun tamamını "yama" sanıyor. 300 adayda 1 gerçek yama.
YAMA_01: kabul 28
YAMA_02: hicbiri
YAMA_03: hicbiri
YAMA_04: hicbiri
YAMA_05: hicbiri
YAMA_06: hicbiri
YAMA_07: hicbiri
YAMA_08: hicbiri
YAMA_09: hicbiri
YAMA_10: hicbiri
# Kaldırım: bordürleri buluyor ama hepsi sağlam. 300 adayda 0 bozuk.
KALDIRIM_01: hicbiri
KALDIRIM_02: hicbiri
KALDIRIM_03: hicbiri
KALDIRIM_04: hicbiri
KALDIRIM_05: hicbiri
KALDIRIM_06: hicbiri
KALDIRIM_07: hicbiri
KALDIRIM_08: hicbiri
KALDIRIM_09: hicbiri
KALDIRIM_10: hicbiri
# v2 hasar kutuları (rastgele örnek): D00 ~%83, D40 ~%67, D20 ~%35, D10 5/8 doğru.
HASAR_D00: red 8,12,20,21,26
HASAR_D10: red 1,3,6
HASAR_D20: kabul 1,3,4,5,6,9,13,16,22,29
HASAR_D40: red 4,6,7,13,14,18
HASAR: D00,D40
# Yüzey: asfalt ~%90, parke ~%75 doğru. Beton sayfası çoğunlukla asfalt, toprak sayfası çoğunlukla tarla/çimen.
YUZEY: asfalt,parke
# Eğitime alınacak yeni sınıflar. Boş = hiçbiri (v4 dört hasar sınıfıyla eğitilir).
# Rögarı denemek için: YENI: ROGAR  (yalnızca ~30 doğrulanmış kutu var, zayıf olur)
YENI:
'''

dosya = INCELEME / 'kararlar.txt'
metin = dosya.read_text(encoding='utf-8') if dosya.exists() else KARARLAR

KARAR, HASAR_KABUL, YUZEY_KABUL, YENI_KABUL = {}, None, None, None
for satir in metin.splitlines():
    satir = satir.split('#')[0].strip()
    if ':' not in satir:
        continue
    ad, _, deger = (s.strip() for s in satir.partition(':'))
    if ad == 'HASAR':
        HASAR_KABUL = [s.strip() for s in deger.split(',') if s.strip()]; continue
    if ad == 'YUZEY':
        YUZEY_KABUL = [s.strip() for s in deger.split(',') if s.strip()]; continue
    if ad == 'YENI':
        YENI_KABUL = [s.strip() for s in deger.split(',') if s.strip()]; continue
    liste = INDEKS['sayfalar'].get(ad)
    if liste is None:
        print('Bilinmeyen sayfa, atlandı:', ad); continue
    kelime, _, sayilar = deger.partition(' ')
    nums = {int(x) for x in sayilar.replace(',', ' ').split() if x.isdigit()}
    for i, a in enumerate(liste, 1):
        dogru = {'hepsi': True, 'hicbiri': False, 'red': i not in nums, 'kabul': i in nums}[kelime.lower()]
        KARAR[(a['dosya'], a['sinif'], tuple(a['kutu']))] = dogru

if not KARAR:
    raise RuntimeError('İnceleme bekleniyor: Claude\'un kararlarını KARARLAR metnine yapıştırıp Tümünü çalıştır deyin.')

def oto_esik(k, hedef=0.85, en_az=10):
    ornek = sorted([(a['skor'], KARAR[(a['dosya'], a['sinif'], tuple(a['kutu']))])
                    for liste in INDEKS['sayfalar'].values() for a in liste
                    if a['sinif'] == k and a['grup'] == 'rastgele' and (a['dosya'], a['sinif'], tuple(a['kutu'])) in KARAR],
                   reverse=True)
    esik = None
    for t, _ in ornek:
        ust = [ok for s, ok in ornek if s >= t]
        if len(ust) >= en_az and sum(ust) / len(ust) >= hedef:
            esik = t
    return esik

OTO_KABUL = {k: oto_esik(k) for k in YENI}
HASAR_KABUL = HASAR_KABUL if HASAR_KABUL is not None else HASAR
YUZEY_KABUL = YUZEY_KABUL if YUZEY_KABUL is not None else []
YENI_KABUL = [k for k in (YENI_KABUL if YENI_KABUL is not None else YENI) if k in YENI]
EGITIM = HASAR + YENI_KABUL     # modelin eğitileceği sınıflar (sıra = sınıf kimliği)
print('İnceleme isabeti (incelenen adaylarda doğru oranı):')
for k in SINIFLAR:
    for g in ('ust', 'rastgele', 'ornek'):
        v = [KARAR[(a['dosya'], a['sinif'], tuple(a['kutu']))] for l in INDEKS['sayfalar'].values() for a in l
             if a['sinif'] == k and a['grup'] == g and (a['dosya'], a['sinif'], tuple(a['kutu'])) in KARAR]
        if v:
            print(f'  {k:9s} {g:9s} {sum(v):4d}/{len(v):<4d} %{100 * sum(v) / len(v):.0f}')
print('Otomatik kabul eşikleri:', OTO_KABUL)
print('Güvenilir hasar sınıfları:', HASAR_KABUL, '| yüzey:', YUZEY_KABUL)
print('Eğitim sınıfları:', EGITIM)

## 8. v4 veri seti

- **RDD2022:** Bölümler v1–v3 ile aynı (seed=42). Eğitim görüntülerine Grounding DINO ile **otomatik kabul eşiğini geçen** yeni sınıf kutuları eklenir. Japonya ve ABD görüntülerinde de rögar ve yama var; etiketlenmezlerse model bunları "arka plan" diye öğrenir.
- **Önceki Türkiye paketleri** (`turkiye_egitim.zip` ×1, `hasar_etiket.zip` ×3) aynı şekilde yeni sınıflarla tamamlanır.
- **Türkiye geneli:** Kabul edilen hasar ve yeni sınıf kutuları eklenir; belirsiz görüntüler dışarıda kalır. Coğrafi sızıntı olmasın diye **her 10. il doğrulama setine** ayrılır.
- Test bölümü yalnızca RDD testidir ve v1–v3 ile aynıdır. Asıl karar yine Bursa test setinde verilir (yerelde).

In [ ]:
if not Path('/content/rdd_yolo/data.yaml').exists():
    !python scripts/download_rdd2022.py --out /content/rdd_raw
    !python scripts/prepare_rdd2022.py --src /content/rdd_raw --out /content/rdd_yolo
RDD = Path('/content/rdd_yolo')
VERI = Path('/content/v4_veri')
shutil.rmtree(VERI, ignore_errors=True)

def yolo_satir(k, b, W, H):
    x1, y1, x2, y2 = b
    x1, x2, y1, y2 = max(0, x1), min(W, x2), max(0, y1), min(H, y2)
    return f'{EGITIM.index(k)} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}'

def yeni_sec(adaylar, dosya, inceleme=True):
    kabul, belirsiz = [], False
    for k, s, b in adaylar.get(dosya, []):
        if k not in YENI_KABUL:   # eğitime alınmayan sınıf: yok say
            continue
        key = (dosya, k, tuple(b))
        if inceleme and key in KARAR:
            if KARAR[key]:
                kabul.append((k, b))
        elif OTO_KABUL.get(k) is not None and s >= OTO_KABUL[k]:
            kabul.append((k, b))
        elif s >= 0.40:
            belirsiz = True
    return kabul, belirsiz

def ekle(src_img, bolum, ad, satirlar):
    (VERI / 'images' / bolum).mkdir(parents=True, exist_ok=True)
    (VERI / 'labels' / bolum).mkdir(parents=True, exist_ok=True)
    dst = VERI / 'images' / bolum / f'{ad}.jpg'
    if not dst.exists():
        os.symlink(src_img, dst)
    (VERI / 'labels' / bolum / f'{ad}.txt').write_text('\n'.join(satirlar))

oto_var = any(OTO_KABUL.get(k) is not None for k in YENI_KABUL)
sayac = Counter()

# 1) RDD2022: orijinal etiketler + (train) otomatik yeni sınıflar
rdd_train = sorted((RDD / 'images' / 'train').glob('*.jpg'))
RDD_YENI = yeni_aday(rdd_train, ETIKET / 'yeni_rdd_train.json') if oto_var else {}
for bolum in ('train', 'val', 'test'):
    for img in sorted((RDD / 'images' / bolum).glob('*.jpg')):
        lbl = RDD / 'labels' / bolum / f'{img.stem}.txt'
        satirlar = [l for l in (lbl.read_text().splitlines() if lbl.exists() else []) if l.strip()]
        if bolum == 'train' and RDD_YENI.get(img.name):
            W, H = Image.open(img).size
            for k, b in yeni_sec(RDD_YENI, img.name, inceleme=False)[0]:
                satirlar.append(yolo_satir(k, b, W, H)); sayac[f'rdd_{k}'] += 1
        ekle(img, bolum, img.stem, satirlar)

# 2) Önceki Türkiye paketleri (yalnızca eğitim)
EK = []
for paket, tekrar in [(Path(DRIVE_DIR) / 'turkiye_egitim.zip', 1)] + [(p, 3) for p in sorted(Path(DRIVE_DIR).glob('hasar_etiket*.zip'))]:
    if not paket.exists():
        continue
    hedef = Path('/content/ek') / paket.stem
    if not hedef.exists():
        with zipfile.ZipFile(paket) as z:
            z.extractall(hedef)
    EK += [(img, hedef / 'labels' / f'{img.stem}.txt', tekrar) for img in sorted((hedef / 'images').glob('*.jpg'))]
EK_YENI = yeni_aday([e[0] for e in EK], ETIKET / 'yeni_onceki_paketler.json') if oto_var else {}
for img, lbl, tekrar in EK:
    satirlar = [l for l in (lbl.read_text().splitlines() if lbl.exists() else []) if l.strip()]
    W, H = Image.open(img).size
    satirlar += [yolo_satir(k, b, W, H) for k, b in yeni_sec(EK_YENI, img.name, inceleme=False)[0]]
    for t in range(tekrar):
        ekle(img, 'train', f'ek_{img.stem}' + (f'_k{t}' if t else ''), satirlar)
    sayac['onceki_paket'] += 1

# 3) Türkiye geneli
iller = sorted({il.split('_')[0] for il in IL_OF.values()})
VAL_ILLER = set(iller[::10])
for img in GORUNTULER:
    h = HASAR_ET[img.name]
    if h['belirsiz'] or any(b[0] not in HASAR_KABUL for b in h['kesin']):
        sayac['tr_atlandi_hasar_belirsiz'] += 1; continue
    yeni, belirsiz = yeni_sec(YENI_ADAY, img.name)
    if belirsiz:
        sayac['tr_atlandi_yeni_belirsiz'] += 1; continue
    W, H = h['boyut']
    satirlar = [yolo_satir(b[0], b[2], W, H) for b in h['kesin'] if KARAR.get((img.name, b[0], tuple(b[2])), True)]
    satirlar += [yolo_satir(k, b, W, H) for k, b in yeni]
    bolum = 'val_tr' if IL_OF[img.name].split('_')[0] in VAL_ILLER else 'train'
    ekle(img, bolum, f'trg_{img.stem}', satirlar)
    if bolum == 'val_tr':
        ekle(img, 'val', f'trg_{img.stem}', satirlar)
    sayac[f'tr_{bolum}'] += 1
    for l in satirlar:
        sayac[f'tr_{EGITIM[int(l.split()[0])]}'] += 1

isimler = ''.join(f'  {i}: {k}\n' for i, k in enumerate(EGITIM))
(VERI / 'data.yaml').write_text(f'path: {VERI}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n{isimler}')
(VERI / 'tr_val.yaml').write_text(f'path: {VERI}\ntrain: images/train\nval: images/val_tr\ntest: images/val_tr\nnames:\n{isimler}')
DATA_V4 = str(VERI / 'data.yaml')
print('Doğrulama illeri:', sorted(VAL_ILLER))
print(json.dumps(dict(sayac), indent=1, ensure_ascii=False))
jsave(dict(sayac), Path(OUT) / 'veri_ozeti.json')

## 9. Ana model v4 (YOLO11m, 800 px)

v2'nin ağırlıklarından devam eder. 7. hücrede `YENI:` boşsa sınıflar v2 ile aynıdır ve bütün katmanlar kaldığı yerden devam eder. Yeni sınıf eklenirse yalnızca son katman yeniden öğrenilir; omurgadaki bilgi korunur.
Eğitim 40 epoch sürer; 10 epoch boyunca iyileşme olmazsa erken durur. Oturum koparsa kaldığı yerden devam eder.

In [ ]:
PROJECT = f'{DRIVE_DIR}/runs'

def egit(baslangic, run_name, **kw):
    last = Path(PROJECT) / run_name / 'weights' / 'last.pt'
    if last.exists():
        try:
            print('Önceki eğitim bulundu, kaldığı yerden devam ediliyor:', last)
            m = YOLO(str(last)); m.train(resume=True)
            return Path(m.trainer.save_dir)
        except AssertionError as e:   # eğitim zaten bitmişse
            print('Eğitim zaten tamamlanmış, sonuçlar kullanılıyor.', e)
            return last.parent.parent
    m = YOLO(str(baslangic))
    m.train(project=PROJECT, name=run_name, exist_ok=True, cos_lr=True, close_mosaic=5, seed=42, plots=True, workers=8, **kw)
    return Path(m.trainer.save_dir)

RUN_V4 = egit(Path(DRIVE_DIR) / 'v2' / 'best_v2.pt', 'v4_yolo11m_800', data=DATA_V4, epochs=40, imgsz=800, batch=16, patience=10)
BEST_V4 = RUN_V4 / 'weights' / 'best.pt'
shutil.copy(BEST_V4, f'{OUT}/best_v4.pt')
print('Kaydedildi:', f'{OUT}/best_v4.pt')

## 10. Telefon / Raspberry Pi modeli v4 (YOLO11n, 640 px) ve yol yüzeyi sınıflandırıcısı

Telefon modeli v3 telefon modelinden devam eder ve ONNX'e çevrilir (telefon tarayıcısı için). Raspberry Pi'de aynı `.pt` dosyası NCNN'e çevrilecek.
Yüzey sınıflandırıcısı (YOLO11n-cls) yalnızca 7. hücrede güvenilir bulunan yüzey türleriyle ve yolun alt-orta kesitiyle eğitilir. Uygulama da aynı kesiti kullanacak.

In [ ]:
EGIT_CANLI = True
if EGIT_CANLI:
    CANLI_DIR = egit(Path(DRIVE_DIR) / 'v3' / 'best_canli.pt', 'v4_yolo11n_640_canli', data=DATA_V4, epochs=60, imgsz=640,
                     batch=32, patience=15)
    BEST_CANLI_V4 = CANLI_DIR / 'weights' / 'best.pt'
    onnx = YOLO(str(BEST_CANLI_V4)).export(format='onnx', imgsz=640, simplify=True, dynamic=False, opset=17)
    shutil.copy(BEST_CANLI_V4, f'{OUT}/best_canli_v4.pt')
    shutil.copy(onnx, f'{OUT}/canli_v4.onnx')
    print('Kaydedildi:', f'{OUT}/canli_v4.onnx')

YUZEY_DIR = Path('/content/yuzey')
if len(YUZEY_KABUL) >= 2:
    shutil.rmtree(YUZEY_DIR, ignore_errors=True)
    r = random.Random(42)
    adlar = {y: [d for d, v in YUZEY_ET.items() if v[0] == y and v[1] >= 0.55] for y in YUZEY_KABUL}
    for y, liste in adlar.items():
        r.shuffle(liste)
        n_train = 0
        il_val = sum(1 for d in liste if IL_OF[d].split('_')[0] in VAL_ILLER) >= 10
        for i, d in enumerate(liste):
            # Doğrulama: ayrılmış illerden; o türden yeterli örnek yoksa rastgele %10
            bolum = 'val' if (IL_OF[d].split('_')[0] in VAL_ILLER if il_val else i % 10 == 0) else 'train'
            if bolum == 'train':
                if n_train >= 1500:   # asfalt baskın: sınıfları dengele
                    continue
                n_train += 1
            (YUZEY_DIR / bolum / y).mkdir(parents=True, exist_ok=True)
            c = yol_kirp(Image.open(HAM / d).convert('RGB'))
            c.thumbnail((320, 320)); c.save(YUZEY_DIR / bolum / y / d, quality=90)
    print({y: len(list((YUZEY_DIR / 'train' / y).glob('*.jpg'))) for y in YUZEY_KABUL})
    YUZEY_RUN = egit('yolo11n-cls.pt', 'v4_yuzey_yolo11n_cls', data=str(YUZEY_DIR), epochs=30, imgsz=224, batch=64, patience=8)
    shutil.copy(YUZEY_RUN / 'weights' / 'best.pt', f'{OUT}/yuzey_v4.pt')
    jsave({'siniflar': YUZEY_KABUL, 'kirpma': {'x': [0.2, 0.8], 'y': [0.55, 0.95]}, 'girdi_boyutu': 224},
          Path(OUT) / 'yuzey_v4.json')
    print('Kaydedildi:', f'{OUT}/yuzey_v4.pt')
else:
    print('Yüzey eğitimi atlandı: güvenilir yüzey türü sayısı < 2', YUZEY_KABUL)

## 11. Değerlendirme

- **RDD2022 testi (4 hasar sınıfı):** v2 ile v4 aynı test görüntülerinde karşılaştırılır. Amaç, yeni sınıflar eklenirken hasar bilgisinin kaybolmadığını göstermek.
- **Türkiye doğrulama illeri (8 sınıf):** Bu ölçüm yarı otomatik etiketlere göre yapılır, kesin bir ölçüm değildir. Yeni sınıfların öğrenilip öğrenilmediğini gösterir.
- **Sınıf eşikleri:** Doğrulama setinde F1'i en yüksek yapan değerler. Uygulama bu eşikleri kullanır.
- Asıl karar Bursa test setinde verilir: Claude'a "v4 hazır" yazın.

In [ ]:
import pandas as pd
from IPython.display import display

def f1_thresholds(metrics, names, lo=0.05, hi=0.9, fallback=0.35):
    b = metrics.box
    out = {}
    for i, cls_id in enumerate(b.ap_class_index):
        f1 = np.asarray(b.f1_curve)[i]
        j = int(f1.argmax())
        esik = float(np.asarray(b.px)[j]) if f1[j] >= 0.1 else fallback
        out[names[int(cls_id)]] = {'esik': round(min(hi, max(lo, esik)), 3), 'f1': round(float(f1[j]), 3)}
    return out

def ozet(m):
    per = {}
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        per[m.names[int(c)]] = round(float(ap50), 3)
    return {'mAP50': round(float(m.box.map50), 3), 'mAP50-95': round(float(m.box.map), 3),
            'precision': round(float(m.box.mp), 3), 'recall': round(float(m.box.mr), 3), 'AP50': per}

MODELLER = {'v2 (YOLO11m, 800)': (f'{DRIVE_DIR}/v2/best_v2.pt', 800, str(RDD / 'data.yaml')),
            'v4 (YOLO11m, 800)': (str(BEST_V4), 800, DATA_V4)}
if EGIT_CANLI:
    MODELLER['canli v4 (YOLO11n, 640)'] = (str(BEST_CANLI_V4), 640, DATA_V4)
sonuc, satirlar = {}, []
for ad, (yol, sz, veri) in MODELLER.items():
    m = YOLO(yol)
    test = m.val(data=veri, split='test', imgsz=sz, batch=16, plots=False, verbose=False)
    sonuc[ad] = {'rdd_test': ozet(test)}
    if 'v4' in ad:
        etiket = 'canli_v4' if 'canli' in ad else 'v4'
        val = m.val(data=DATA_V4, split='val', imgsz=sz, batch=16, plots=False, verbose=False)
        trv = m.val(data=str(VERI / 'tr_val.yaml'), split='val', imgsz=sz, batch=16, plots=True, verbose=False,
                    project=OUT, name=f'{etiket}_tr_dogrulama', exist_ok=True)
        th = f1_thresholds(val, m.names)
        sonuc[ad].update({'tr_dogrulama': ozet(trv), 'esikler': th})
        icerik = {'esikler': {c: v['esik'] for c, v in th.items()},
                  'kaynak': f'v4 doğrulama seti (RDD + Türkiye), sınıf başına en yüksek F1 ({ad})'}
        if 'canli' in ad:
            icerik['girdi_boyutu'] = 640
        jsave(icerik, Path(OUT) / f'esikler_{etiket}.json')
    satirlar.append({'model': ad, 'RDD mAP50': sonuc[ad]['rdd_test']['mAP50'],
                     **{f'RDD {c}': v for c, v in sonuc[ad]['rdd_test']['AP50'].items()},
                     **({f'TR {c}': v for c, v in sonuc[ad]['tr_dogrulama']['AP50'].items()} if 'tr_dogrulama' in sonuc[ad] else {})})

display(pd.DataFrame(satirlar).set_index('model'))
jsave(sonuc, Path(OUT) / 'kiyas_v2_v4.json')
print('\nDrive:', OUT)
!ls -la {OUT}

## Sonra yapılacaklar
Claude'a **"v4 hazır"** yazın. Claude şunları yapacak:
1. `best_v4.pt`, `best_canli_v4.pt`, `canli_v4.onnx`, `yuzey_v4.pt` ve eşik dosyalarını indirecek.
2. v2 ile v4'ü **Bursa test setinde** ölçecek. Türkiye verisi yakalamayı artırıp yanlış alarmı düşük tutuyor mu, bakacak.
3. Daha iyi olan modeli uygulamaya alacak: etiket aracı, şiddet ağırlıkları, telefon modeli ve Raspberry Pi.